In [ ]:
#================================================================================
# 📚 데이터셋 정보: amphora/mmlu-korean-translated
# 💡 의미: 한국어 번역된 MMLU(Massive Multitask Language Understanding) 테스트 데이터셋
# 📖 설명: 다양한 학술적 분야에 걸친 객관식 질문과 답변을 포함하는 데이터셋입니다.
# 이 데이터를 활용하여 초보자도 할 수 있는 데이터 탐색, 분석, 그리고
# AI 모델 학습을 위한 프롬프트 구조화 실습을 해볼 거예요!
#================================================================================

import random
from datasets import load_dataset, Dataset
import time
import json

# --- [설정] ---
DATASET_NAME = "amphora/mmlu-korean-translated"
DATASET_SPLIT = "test"
SAMPLE_COUNT = 10  # 처음 실습할 샘플 개수 (메모리 절약 및 빠른 진행을 위해 적게 설정!)

print("✨ AI 튜터: 안녕하세요! 초보 파이썬 코딩을 위한 아주 신나는 데이터 탐험을 시작해 볼까요? 😊")
print("🚀 이 데이터셋은 '질문(question)'과 4개의 보기(A, B, C, D), 그리고 정답(answer)'으로 구성된 AI 지식 퀴즈 모음이에요!")
print("=" * 80)


# -------------------------------------------------------------------
# 🛠️ 1. 데이터 로딩: 스트리밍 VS 일반 모드 처리 (가장 중요!)
# -------------------------------------------------------------------

dataset = None
try:
    # 1단계: 스트리밍 모드로 시도 (가장 빠르고 메모리 효율적!)
    print(f"\n👉 [1/4] 데이터를 스트리밍 모드(streaming=True)로 로드해 보겠습니다. (OOM 방지 필수!)")
    start_time = time.time()
    dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT, streaming=True)
    print(f"✅ 로딩 성공! (소요 시간: {time.time() - start_time:.2f}초)")

except Exception as e:
    # 스트리밍 로드에 실패할 경우 (일부 환경에서 문제 발생 가능)
    print(f"\n⚠️ 스트리밍 로드에 실패했습니다. ({e.__class__.__name__} 에러 감지)")
    print("🔄 대신, 소량만 다운로드하여 일반 데이터셋(Dataset)으로 진행하겠습니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT, streaming=False)
    except Exception as e_fallback:
        print(f"❌ 치명적인 오류 발생: {e_fallback}. 데이터 로드가 불가능합니다.")
        exit()


# -------------------------------------------------------------------
# 🔄 2. 샘플 데이터 추출 (핵심 패턴 적용)
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print(f"✨ 2. {DATASET_NAME} 데이터에서 상위 {SAMPLE_COUNT}개 샘플만 추출합니다.")

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    print("💻 스트리밍 모드 감지: .take() 메서드를 사용합니다.")
    sample_iterator = dataset.take(SAMPLE_COUNT)
    # Iterator를 리스트로 변환하여 반복 가능하도록 만듭니다.
    sample_data_list = list(sample_iterator) 
else:
    # 일반 데이터셋 (Dataset)
    print("💾 일반 데이터셋 감지: 리스트 슬라이싱을 사용합니다.")
    sample_data_list = dataset.select(range(min(SAMPLE_COUNT, len(dataset))))


# -------------------------------------------------------------------
# 🔎 3. 데이터 구조 분석 및 기초 탐색 실습 (Data Exploration)
# -------------------------------------------------------------------

print("\n\n💖 3. 데이터 구조 탐색: 이 데이터가 어떤 옷을 입고 있는지 살펴보기!")

if sample_data_list:
    print(f"📂 분석할 데이터 개수: {len(sample_data_list)}개 (전체 {DATASET_SPLIT} 세트 중 상위 {SAMPLE_COUNT}개)")
    
    # 첫 번째 샘플을 분석하여 구조를 파악합니다.
    sample_0 = sample_data_list[0]
    print("-" * 40)
    print(f"📝 [첫 번째 샘플] 질문(Question): {sample_0['question'][:40]}... (상위 40자)")
    print(f"🏷️ 카테고리(Category): {sample_0['category']}")
    print(f"✅ 정답(Answer): {sample_0['answer']} (정답 인덱스/값)")
    print("-" * 40)

    # 💡 Tip: 데이터셋의 컬럼(Feature) 이름 확인하기
    print("\n📊 데이터셋의 모든 항목(Key) 이름:", list(sample_0.keys()))
    
    # 📈 정량적 분석: 카테고리 분포 분석
    print("\n🔍 [분석] 카테고리 분포 살펴보기 (가장 많이 나온 주제는?)")
    category_counts = {}
    for sample in sample_data_list:
        category = sample.get('category', 'N/A')
        category_counts[category] = category_counts.get(category, 0) + 1
    
    # 가장 많이 등장한 카테고리를 찾습니다.
    most_common_category = max(category_counts, key=category_counts.get)
    print(f"✨ 가장 많이 등장한 카테고리: '{most_common_category}' (출현 횟수: {category_counts[most_common_category]}회)")

else:
    print("❌ 경고: 샘플 데이터가 없어 분석을 진행할 수 없습니다.")


# -------------------------------------------------------------------
# 🧠 4. AI 실습: LLM 프롬프트 생성 및 데이터 변환 시뮬레이션
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("🚀 4. AI 시뮬레이션: 모델에게 질문을 던지는 프롬프트 만들기 연습!")

# 이 데이터셋을 LLM(거대 언어 모델)에 넣어서 학습시키거나 테스트할 때,
# 모델이 이해하기 쉬운 형태로 데이터를 가공하는 과정이 필요합니다.

processed_qa_list = []
for i, sample in enumerate(sample_data_list):
    if i >= 5: # 과부하 방지 및 예시 간결화
        break
        
    # 목표: '문제' + '보기들' + '정답 안내'를 하나의 문자열로 조합
    question = sample['question']
    
    # ⚠️ 데이터 형태가 문자열/직접 사용하기 좋은지 확인
    options = f"A) {sample['A']} | B) {sample['B']} | C) {sample['C']} | D) {sample['D']}"
    
    # 정답은 인덱스(answer)가 아니라, 보기에 들어있는 실제 정답 문자열을 꺼내는 게 좋아요.
    # 여기서는 정답을 '정답은 {sample['answer']}번입니다.' 와 같이 설명해 줍니다.
    correct_answer_hint = f"정답: {sample['answer']}번"
    
    # 최종 프롬프트 구조화
    llm_prompt = (
        f"질문: {question}\n"
        f"선택지: {options}\n"
        f"답변 요청: 위의 질문을 풀고 정답을 골라주세요.\n"
        f"[채점 가이드] {correct_answer_hint}"
    )
    
    processed_qa_list.append({
        "Prompt": llm_prompt,
        "Original_Category": sample['category']
    })

print("\n🌟 [결과] 모델 학습용 프롬프트 구조화 결과 (상위 5개만 출력):")
for i, qa_item in enumerate(processed_qa_list):
    print(f"--- 샘플 {i+1} (카테고리: {qa_item['Original_Category']}) ---")
    print(qa_item['Prompt'])

print("\n🎉 축하합니다! 데이터셋 구조를 완벽하게 파악하고, AI 모델이 이해하기 쉬운 형태로 데이터를 변환하는 실습을 성공적으로 완료했습니다!")
print("👩‍🏫 다음 단계: 이 'Prompt' 구조를 기반으로, 파이썬 코드를 이용해 실제 예측 모델을 만들어보는 것이 목표가 될 거예요!")